In [9]:
import boto3
import json
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime
import os
from dotenv import load_dotenv
import re
import uuid
import mlflow
import dagshub

# Load env file
load_dotenv()

os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("DAGSHUB_TOKEN")

# Tracking server
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("inframind_experiment")

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name="ap-south-1",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

In [10]:
class RCAOutput(BaseModel):
    incident_id: str = Field(description="Unique ID for the incident")
    severity: str = Field(description="Critical, High, Medium, or Low")
    summary: str = Field(description="One-sentence summary of what happened")
    root_cause: str = Field(description="The primary technical reason for the failure")
    immediate_fix: str = Field(description="What to do right now to restore service")
    confidence_score: float = Field(description="AI's confidence (0.0 to 1.0)")
    model_used: str = Field(description="Llama-3-8B or Llama-3-70B")

json_schema = RCAOutput.model_json_schema()

In [11]:
def generate_infra_rca(raw_log: str, context: str = "") -> RCAOutput:

    # 0. Generate a unique incident ID for tracking
    incident_id = str(uuid.uuid4())

    # 1. Simple Routing Logic
    # If the log is massive, we escalate to the 70B model
    selected_model = "meta.llama3-70b-instruct-v1:0" if len(raw_log) > 2000 else "meta.llama3-8b-instruct-v1:0"
    
    # 2. Construct the Prompt
    prompt = f"""
    <system>
    You are a Senior SRE. 
    The incident ID is: {incident_id}
    Analyze the log and return ONLY valid JSON. Do not include explanations, markdown, or text outside JSON.
    Format your response strictly according to this JSON schema: {json.dumps(json_schema)}
    You MUST include the provided incident_id in the JSON.
    </system>
    
    <incident_context>
    {context}
    </incident_context>
    
    <log>
    {raw_log}
    </log>
    
    JSON Output:
    """

    # 3. Call AWS Bedrock
    response = bedrock_runtime.invoke_model(
        modelId=selected_model,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt": prompt,
            "max_gen_len": 1024,
            "temperature": 0.1,
            "top_p": 0.9,
            "stop": ["\n\n"]
        })
    )
    
    # 4. Parse and Validate
    result = json.loads(response["body"].read())
    generation = result["generation"]

    # Extract JSON block
    def extract_json(text: str) -> str:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1:
            raise ValueError("No JSON object found in model output")
        return text[start:end + 1]

    # Extract JSON safely
    json_text = extract_json(generation)

    try:
        rca = RCAOutput.model_validate_json(json_text)
    except Exception as e:
        raise ValueError(f"Model returned invalid JSON: {json_text}") from e

    # Force correct metadata
    rca.incident_id = incident_id
    model_label = "Llama-3-70B" if "70b" in selected_model else "Llama-3-8B"
    rca.model_used = model_label

    return rca

In [12]:
test_log = "ERROR: [Kubelet] Failed to pull image 'nginx:latest': RPC error: code = Unknown desc = Error response from daemon: Get https://registry-1.docker.io/v2/: net/http: TLS handshake timeout"

try:
    analysis = generate_infra_rca(test_log, context="VPC Security Groups allow outbound 443.")
    print("\n--- RCA REPORT ---")
    print("Incident ID:", analysis.incident_id)
    print("Severity:", analysis.severity)
    print("Summary:", analysis.summary)
    print("Root Cause:", analysis.root_cause)
    print("Immediate Fix:", analysis.immediate_fix)
    print("Confidence:", analysis.confidence_score)
    print("Model:", analysis.model_used)
except Exception as e:
    print(f"Mission Failed: {e}")


--- RCA REPORT ---
Incident ID: 2a090391-3288-497b-b29c-c7e15ff89ba1
Severity: High
Summary: Failed to pull image 'nginx:latest' due to TLS handshake timeout
Root Cause: TLS handshake timeout
Immediate Fix: Check VPC Security Groups for outbound 443
Confidence: 0.8
Model: Llama-3-8B


In [13]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings

# Load documents
loader = DirectoryLoader("runbook", glob="*.md", loader_cls=TextLoader)
docs = loader.load()

print("Docs:", len(docs))

headers_to_split_on = [
    ("#", "Header1"),
    ("##", "Header2"),
    ("###", "Header3"),
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

splits = []

for doc in docs:
    splits.extend(splitter.split_text(doc.page_content))

print("Splits:", len(splits))

embeddings = BedrockEmbeddings(
    client=bedrock_runtime,
    model_id="amazon.titan-embed-text-v2:0"
)

vector_db = FAISS.from_documents(
    splits,
    embeddings
)

def get_context(query):
    results = vector_db.similarity_search(query, k=4)
    return "\n---\n".join([doc.page_content for doc in results])

Docs: 3
Splits: 41


In [14]:
# 1. Define the incident
new_incident_log = "ERROR 500: Database connection refused while connecting to host rds.cluster-inframind.internal"

with mlflow.start_run():

    # Log input
    mlflow.log_param("incident_log", new_incident_log)

    # Retrieve knowledge
    retrieved_knowledge = get_context(
        new_incident_log + " root cause fix"
    )

    mlflow.log_param("retrieved_context_length", len(retrieved_knowledge))

    # Generate RCA
    final_rca = generate_infra_rca(new_incident_log, context=retrieved_knowledge)

    # Log model info
    mlflow.log_param("model_used", final_rca.model_used)

    # Log RCA outputs
    mlflow.log_metric("confidence_score", final_rca.confidence_score)
    mlflow.log_text(retrieved_knowledge, "retrieved_context.txt")
    
    mlflow.log_dict({
        "incident_id": final_rca.incident_id,
        "severity": final_rca.severity,
        "summary": final_rca.summary,
        "root_cause": final_rca.root_cause,
        "immediate_fix": final_rca.immediate_fix
    }, "rca_output.json")

    # Print results
    print("\n--- FULL ANALYTICS REPORT ---")
    print("Incident ID:", final_rca.incident_id)
    print("Knowledge Used:\n", retrieved_knowledge)
    print("Severity:", final_rca.severity)
    print("Summary:", final_rca.summary)
    print("Root Cause:", final_rca.root_cause)
    print("Immediate Fix:", final_rca.immediate_fix)
    print("Confidence:", final_rca.confidence_score)
    print("Model:", final_rca.model_used)


--- FULL ANALYTICS REPORT ---
Incident ID: 1d05097c-cf63-4edd-b713-c79ea1bd449b
Knowledge Used:
 **Symptoms:**
- `connection timeout` errors in logs
- Applications unable to connect to database
- High connection pool exhaustion  
**Immediate Actions:**
1. Check database server status: `kubectl get pods -l app=database`
2. Verify connection limits: `SHOW VARIABLES LIKE 'max_connections';`
3. Kill long-running queries: `SHOW PROCESSLIST;`  
**Root Cause Analysis:**
- Connection pool misconfiguration
- Unoptimized queries causing locks
- Insufficient database resources
---
**Symptoms:**
- `connection refused` errors between services
- Intermittent connectivity issues
- DNS resolution failures  
**Immediate Actions:**
1. Check network policies: `kubectl get networkpolicy -A`
2. Verify pod connectivity: `kubectl exec -it pod -- nc -zv target-service 80`
3. Test DNS resolution: `kubectl exec -it pod -- nslookup service-name`  
**Root Cause Analysis:**
- Restrictive network policies blocking

In [8]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

# Since DeepEval usually uses OpenAI, we tell it to use your Bedrock setup for grading
def run_deepeval(log, context, output_json):
    # 1. Setup the Test Case
    test_case = LLMTestCase(
        input=log,
        actual_output=output_json.immediate_fix,
        retrieval_context=[context]
    )
    
    # 2. Initialize Metrics (Threshold 0.7 means 70% accuracy required)
    # Faithfulness: Did it stick to the Runbook?
    faith_metric = FaithfulnessMetric(threshold=0.7)
    # Relevancy: Did it actually address the error?
    rel_metric = AnswerRelevancyMetric(threshold=0.7)
    
    # 3. Measure (Note: This might require setting up a 'DeepEval' model wrapper)
    faith_metric.measure(test_case)
    rel_metric.measure(test_case)
    
    return faith_metric.score, rel_metric.score

In [ ]:
def senior_sre_critique(rca_report: RCAOutput, runbook_context: str):
    """The 70B model acts as a Senior SRE to find flaws in the 8B's output."""
    
    critique_prompt = f"""
    <system>
    You are a Staff SRE at a Tier-1 Tech Company. Review the following RCA Report for technical accuracy.
    Check if the 'Immediate Fix' matches the provided Runbook and if the reasoning is sound.
    Be harsh and precise.
    </system>

    <runbook_context>
    {runbook_context}
    </runbook_context>

    <proposed_rca>
    {rca_report.json()}
    </proposed_rca>

    Provide a 'Critique Score' (0-10) and a brief 'Reviewer Note'.
    Format: SCORE: [X] | NOTE: [Reasoning]
    """

    response = bedrock_runtime.invoke_model(
        modelId="meta.llama3-70b-instruct-v1:0",
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "prompt": critique_prompt,
            "max_gen_len": 512,
            "temperature": 0.1
        })
    )
    
    critique_text = json.loads(response['body'].read())['generation']
    return critique_text

In [ ]:
def analyze_log_and_evaluate(test_log, context):
    with mlflow.start_run(run_name=f"EVAL_{datetime.now().strftime('%H%M%S')}"):
        
        # 1. Generate RCA (8B Model)
        rca = generate_infra_rca(test_log, context)
        
        # 2. Get Critique (70B Model)
        critique = senior_sre_critique(rca, context)
        
        # 3. Extract Score (Simple regex to get the number from critique)
        critique_score = float(re.search(r"SCORE: \[(\d+)\]", critique).group(1)) / 10
        
        # 4. Log everything to DagsHub
        mlflow.log_param("log_text", test_log[:50] + "...")
        mlflow.log_metric("senior_sre_score", critique_score)
        mlflow.set_tag("critique_note", critique.split("|")[1])
        
        print(f"Critique Received: {critique}")
        return rca, critique